# CP6 - Operaciones Avanzadas II

### Como siempre, antes de empezar...

Si has cerrado el proyecto o has abierto `Nueva Ventana` en VS Code, pulsa en el primer icono del menú de la izquierda `Explorador` y después en `Abrir Carpeta`. 

Busca la ruta en la que guardas el repositorio de GitHub en tu PC, normalmente *`Documentos => GitHub => fundamentos_big_data` * y pulsa en `Abrir`. De esta forma, ya tendrás la carpeta de tu repositorio cargada en VS Code.

> ¡Importante! Antes de abrir esa carpeta, debes ir a GitHub Desktop y realizar un `pull` para asegurarte de que tus archivos están actualizados. También puedes hacer el pull desde VS Code.  

### Requisitos para arrancar el entonro
- Iniciar Docker (abrir docker desktop).
- Iniciar GitHub (abrir github desktop) y estar logueado.
- En el terminal, ejecutar el siguiente comando en la raíz del proyecto `fundamentos_big_data %` 
```bash
docker-compose build --no-cache
docker-compose up
```
> ¡NO TE OLVIDES! Docker debe estar abierto y en ejecución en tu pc para poder utilizarlo. 


### Seleccionar Kernel
- Seleccionar kernel
- Servidor de Juypter existente
- Escriba la dirección URL del servidor de Jupyter en ejecución
- Seleccionar http://127.0.0.1:8888/?token=xxxxxxxx y pulsar Enter
- 127.0.0.1 y pulsar Enter
- Seleccionar Python 3 (ipykernel)

___________________

## Objetivos de la sesión

- Funciones para el manejo de strings.
- Funciones para trabajar con colecciones.
- Otras funciones y UDF.
- Particionamiento de DataFrames, shuffle y optimización básica.


In [1]:
### Importar las librerías necesarias ###
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window
### INICIA LA SPARK-SESSION ###
spark = SparkSession.builder \
    .appName("PySpark - Operaciones avanzadas 2") \
    .getOrCreate()
### Añade el spark context ###
sc = spark.sparkContext
### Añade el path para acceder a los datos y nómbralo como DATA_PATH###
DATA_PATH = "/home/jovyan/data/"
### Lee la carpeta vuelos con pyspark y nómbralo como df_v ###
df_v = spark.read.parquet(DATA_PATH + "vuelos")

## 3 - Funciones definidas por el usuario (UDF)

**FORMA 1**: Vamos a ver la primera forma de registrar funciones `udf` en spark.

In [2]:
def cubo(n):
  return n**3

Para registrar una función `udf`, tenemos que usar la estructura `spark.udf.register()`. Los parámetros que le pasaremos serán:

- Nombre de la udf.
- Nombre de la función previamente definida.
- Tipo de dato que va a devolver.

In [3]:
spark.udf.register('cubo', cubo, T.LongType())

<function __main__.cubo(n)>

In [4]:
df_u = spark.range(1,10) # creamos un df numérico con range()

In [5]:
df_u.show()

+---+
| id|
+---+
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



Una vez que tenemos registrada nuestra función, ya la podemos integrar. En este caso, le pasamos la columna con los enteros del df creado anteriormente.

In [6]:
df_u.select(
    F.col('id'),
    cubo(F.col('id')).alias('cubo')
).show()

+---+-----+
| id| cubo|
+---+-----+
|  1|  1.0|
|  2|  8.0|
|  3| 27.0|
|  4| 64.0|
|  5|125.0|
|  6|216.0|
|  7|343.0|
|  8|512.0|
|  9|729.0|
+---+-----+



**FORMA 2**: Vamos a ver la segunda forma de crear funciones `udf`.

In [7]:
def bienvenida(nombre):
    return ('Hola, {}'.format(nombre))

In [8]:
bienvenida_udf = F.udf(lambda x: bienvenida(x), T.StringType())

In [9]:
df_u2 = spark.createDataFrame([('Jose',),('Julia',),('Katia',)], ['nombre'])

In [10]:
df_u2.show()

+------+
|nombre|
+------+
|  Jose|
| Julia|
| Katia|
+------+



In [11]:
df_u2.select(
    F.col('nombre'),
    bienvenida_udf(F.col('nombre')).alias('Saludo')
).show(truncate=False)

+------+-----------+
|nombre|Saludo     |
+------+-----------+
|Jose  |Hola, Jose |
|Julia |Hola, Julia|
|Katia |Hola, Katia|
+------+-----------+



**FORMA 3**: Vamos a ver la tercera forma de crear funciones `udf`. Se trata de usar un decorador.

In [12]:
@F.udf(returnType=T.StringType())

def mayuscula(s):
  return s.upper()

In [13]:
df_u2.select(
    F.col('nombre'),
    mayuscula(F.col('nombre')).alias('mayuscula')
).show(truncate=False)

+------+---------+
|nombre|mayuscula|
+------+---------+
|Jose  |JOSE     |
|Julia |JULIA    |
|Katia |KATIA    |
+------+---------+



## 4 - Funciones de Ventana

In [14]:
df_w = spark.read.parquet(DATA_PATH + 'ventanas')

In [15]:
df_w.show()

+------+------------+------+
|nombre|departamento|puntos|
+------+------------+------+
|Lazaro|      letras|    45|
|  Raul|  matemática|    42|
| Maria|  matemática|    31|
|  Jose|     química|    33|
|Susana|     química|    34|
|  Juan|      letras|    37|
| Julia|      letras|    30|
| Kadir|arquitectura|    21|
|Lilian|arquitectura|    42|
+------+------------+------+



Para crear una ventana (`Window`) tenemos que especificarle por qué columna vamos a particionar los datos y en función de qué columna vamos a ordenar la partición.

In [16]:
### Departamento es el por la que asignamos las ventanas
### Tener la ventana ordenada por puntos
windowSpec = Window.partitionBy('departamento').orderBy('puntos')

Una vez hemos creado la ventana, podemos crear un sistema de ordenación por número de fila para cada partición. Para ello usamos `withColumn` y `row_number`, además del método `over()` para pasarle la ventana.

In [17]:
df_w.withColumn('row_number', F.row_number()\
    .over(windowSpec)).show(truncate=False)

+------+------------+------+----------+
|nombre|departamento|puntos|row_number|
+------+------------+------+----------+
|Kadir |arquitectura|21    |1         |
|Lilian|arquitectura|42    |2         |
|Julia |letras      |30    |1         |
|Juan  |letras      |37    |2         |
|Lazaro|letras      |45    |3         |
|Maria |matemática  |31    |1         |
|Raul  |matemática  |42    |2         |
|Jose  |química     |33    |1         |
|Susana|química     |34    |2         |
+------+------------+------+----------+



**Agregaciones con funciones de Ventana**

In [18]:
winSpecAgg = Window.partitionBy('departamento')

In [19]:
(df_w.withColumn('row_number', F.row_number().over(windowSpec)) # ventana ordenada
    .withColumn('min', F.min(F.col('puntos')).over(winSpecAgg)) # ventana agregada
    .withColumn('max', F.max(F.col('puntos')).over(winSpecAgg)) 
    .withColumn('avg', F.avg(F.col('puntos')).over(winSpecAgg))
).show()

+------+------------+------+----------+---+---+------------------+
|nombre|departamento|puntos|row_number|min|max|               avg|
+------+------------+------+----------+---+---+------------------+
| Kadir|arquitectura|    21|         1| 21| 42|              31.5|
|Lilian|arquitectura|    42|         2| 21| 42|              31.5|
| Julia|      letras|    30|         1| 30| 45|37.333333333333336|
|  Juan|      letras|    37|         2| 30| 45|37.333333333333336|
|Lazaro|      letras|    45|         3| 30| 45|37.333333333333336|
| Maria|  matemática|    31|         1| 31| 42|              36.5|
|  Raul|  matemática|    42|         2| 31| 42|              36.5|
|  Jose|     química|    33|         1| 33| 34|              33.5|
|Susana|     química|    34|         2| 33| 34|              33.5|
+------+------------+------+----------+---+---+------------------+



Como aparecen elementos duplicados, es necesario añadir una condición `where` para filtrar por `row_number`.

In [20]:
(df_w.withColumn('row_number', F.row_number().over(windowSpec)) # ventana ordenada
    .withColumn('min', F.min(F.col('puntos')).over(winSpecAgg)) # ventana agregada
    .withColumn('max', F.max(F.col('puntos')).over(winSpecAgg))
    .withColumn('avg', F.avg(F.col('puntos')).over(winSpecAgg))
).where(F.col('row_number') == 1).select(
    F.col('departamento'), F.col('min'), F.col('max'), F.col('avg')
).show()

+------------+---+---+------------------+
|departamento|min|max|               avg|
+------------+---+---+------------------+
|arquitectura| 21| 42|              31.5|
|      letras| 30| 45|37.333333333333336|
|  matemática| 31| 42|              36.5|
|     química| 33| 34|              33.5|
+------------+---+---+------------------+



## 5 - Particiones y optimización en PySpark

Una **partición** es una unidad lógica de distribución de los datos. Las particiones determinan el paralelismo de las tareas, pero **no equivalen necesariamente a ejecutores ni a archivos**. En los ejemplos siguientes usamos datos sintéticos, sin depender de los archivos de la práctica.

> **Importante:** `F.coalesce(col1, col2)` (visto en el apartado 2.15) devuelve el primer valor no nulo entre columnas. Es distinto de `df.coalesce(n)`, que reduce el número de particiones.

In [21]:
# Dataset reproducible para explorar particiones

df_part = (spark.range(0, 120, 1, numPartitions=4)
    .withColumn("grupo", F.when(F.col("id") < 90, "A").otherwise("B"))
    .withColumn("importe", (F.col("id") * 1.5).cast("double")))
print("Particiones iniciales:", df_part.rdd.getNumPartitions())
df_part.groupBy(F.spark_partition_id().alias("particion")).count().orderBy("particion").show()

Particiones iniciales: 4
+---------+-----+
|particion|count|
+---------+-----+
|        0|   30|
|        1|   30|
|        2|   30|
|        3|   30|
+---------+-----+



### 5.1 Consultar particiones y observar su distribución

`df.rdd.getNumPartitions()` permite consultar cuántas particiones tiene el DataFrame; `F.spark_partition_id()` muestra la partición asignada a cada fila durante una ejecución. El reparto puede cambiar si cambia el plan físico o interviene AQE.

In [22]:
df_part.select("id", "grupo", F.spark_partition_id().alias("particion")).show(12)
print("Número de particiones:", df_part.rdd.getNumPartitions())

+---+-----+---------+
| id|grupo|particion|
+---+-----+---------+
|  0|    A|        0|
|  1|    A|        0|
|  2|    A|        0|
|  3|    A|        0|
|  4|    A|        0|
|  5|    A|        0|
|  6|    A|        0|
|  7|    A|        0|
|  8|    A|        0|
|  9|    A|        0|
| 10|    A|        0|
| 11|    A|        0|
+---+-----+---------+
only showing top 12 rows

Número de particiones: 4


### 5.2 `repartition(n)` y `repartition(n, columna)`

`repartition()` crea una nueva distribución de los datos y **normalmente implica un shuffle** (intercambio de datos entre tareas). Reparticionar por columna agrupa valores iguales según la distribución hash, pero **no garantiza tamaños equilibrados** cuando existe *skew*.

In [23]:
df_repartido = df_part.repartition(8)
print("Repartition(8):", df_repartido.rdd.getNumPartitions())
df_repartido.groupBy(F.spark_partition_id().alias("particion")).count().orderBy("particion").show()

df_por_grupo = df_part.repartition(4, "grupo")
print("Repartition(4, 'grupo'):", df_por_grupo.rdd.getNumPartitions())
df_por_grupo.groupBy(F.spark_partition_id().alias("particion"), "grupo").count().orderBy("particion", "grupo").show()

Repartition(8): 8
+---------+-----+
|particion|count|
+---------+-----+
|        0|   14|
|        1|   14|
|        2|   16|
|        3|   16|
|        4|   15|
|        5|   15|
|        6|   15|
|        7|   15|
+---------+-----+

Repartition(4, 'grupo'): 4
+---------+-----+-----+
|particion|grupo|count|
+---------+-----+-----+
|        1|    B|   30|
|        2|    A|   90|
+---------+-----+-----+



### 5.3 `coalesce(n)`: reducir particiones

`df.coalesce(n)` se utiliza para **reducir** particiones sin un shuffle completo en el caso habitual. Puede dejar particiones desiguales. Si se pide un número mayor que el actual, no incrementa las particiones; para aumentarlas utiliza `repartition()`.

In [24]:
df_reducido = df_part.coalesce(2)
print("Coalesce(2):", df_reducido.rdd.getNumPartitions())
df_reducido.groupBy(F.spark_partition_id().alias("particion")).count().orderBy("particion").show()

print("Coalesce(10) sobre 4:", df_part.coalesce(10).rdd.getNumPartitions())

Coalesce(2): 2
+---------+-----+
|particion|count|
+---------+-----+
|        0|   60|
|        1|   60|
+---------+-----+

Coalesce(10) sobre 4: 4


### 5.4 ¿Volver a cero particiones?

No existe una operación válida `df.repartition(0)` o `df.coalesce(0)` para dejar un DataFrame con datos y **cero particiones**. El mínimo práctico para concentrar los registros es **una partición**: `coalesce(1)` o `repartition(1)`. Ambas opciones reducen el paralelismo y pueden generar cuellos de botella; úsalas solo cuando esté justificado, por ejemplo en salidas pequeñas. **Vaciar un DataFrame (cero filas) no es lo mismo que establecer cero particiones.**

In [25]:
df_una = df_part.coalesce(1)
print("Coalesce(1):", df_una.rdd.getNumPartitions())
print("Repartition(1):", df_part.repartition(1).rdd.getNumPartitions())

df_vacio = df_part.filter(F.lit(False))
print("Filas del DataFrame vacío:", df_vacio.count())
print("Particiones del DataFrame vacío:", df_vacio.rdd.getNumPartitions())

# No ejecutar: ambas instrucciones son inválidas
# df_part.repartition(0)
# df_part.coalesce(0)

Coalesce(1): 1
Repartition(1): 1
Filas del DataFrame vacío: 0
Particiones del DataFrame vacío: 0


### 5.5 `sortWithinPartitions()` y `mapPartitions()`

`sortWithinPartitions()` ordena **dentro** de cada partición, no globalmente. `mapPartitions()` pertenece a la API RDD y permite aplicar una función a cada iterador de partición; conviene preferir funciones nativas de DataFrame cuando estén disponibles.

In [26]:
df_ordenado = df_part.repartition(3).sortWithinPartitions("importe")
df_ordenado.select("id", "importe", F.spark_partition_id().alias("particion")).show(12)

# Ejemplo didáctico con RDD: contar registros por partición
conteos = df_part.rdd.mapPartitions(lambda filas: [sum(1 for _ in filas)]).collect()
print("Registros por partición:", conteos)

+---+-------+---------+
| id|importe|particion|
+---+-------+---------+
|  2|    3.0|        0|
|  3|    4.5|        0|
|  7|   10.5|        0|
|  9|   13.5|        0|
| 13|   19.5|        0|
| 17|   25.5|        0|
| 19|   28.5|        0|
| 20|   30.0|        0|
| 24|   36.0|        0|
| 28|   42.0|        0|
| 32|   48.0|        0|
| 36|   54.0|        0|
+---+-------+---------+
only showing top 12 rows

Registros por partición: [30, 30, 30, 30]


### 5.6 `partitionBy()` al escribir: particiones físicas de almacenamiento

`DataFrameWriter.partitionBy("grupo")` organiza la salida en **directorios** por valor de columna. Es diferente de `repartition()`, que organiza las particiones de ejecución. Se pueden combinar, pero no son equivalentes.

In [27]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory(prefix="spark_particiones_") as ruta:
    (df_part.write.mode("overwrite").partitionBy("grupo").parquet(ruta))
    print("Directorios creados:", sorted(p.name for p in Path(ruta).iterdir() if p.is_dir()))
    print("Filas recuperadas:", spark.read.parquet(ruta).count())

Directorios creados: ['grupo=A', 'grupo=B']
Filas recuperadas: 120


### 5.7 Caché y persistencia

`cache()` y `persist()` ayudan cuando un DataFrame se reutiliza en varias acciones. La caché es **perezosa**: se materializa con una acción. Usa `unpersist()` cuando termines para liberar recursos.

In [29]:
df_cache = df_part.repartition(4).cache()
print("Filas materializadas en caché:", df_cache.count())
print("Suma de importes:", df_cache.agg(F.sum("importe")).first()[0])
df_cache.unpersist()

Filas materializadas en caché: 120
Suma de importes: 10710.0


DataFrame[id: bigint, grupo: string, importe: double]

## Ejercicios

Vamos a utilizar los datos de vuelos. Lo primero que debes hacer es leer el dataset como `df` o `df_v`

**Soluciones:** cada ejercicio está resuelto en la celda de código inmediatamente posterior. Ejecuta primero la celda de configuración que crea `spark` y carga `df_v`. Los resultados numéricos dependen de los datos disponibles en `/home/jovyan/data/vuelos`. 

#### **Ejercicio 1 - Retraso medio y desviación por aerolínea y mes**

Para vuelos no cancelados, calcula `avg_departure_delay` por `AIRLINE` y `MONTH`. Ordena por mayor retraso medio.

**Resultado esperado (formato + ejemplo):**
```
+--------+-----+--------------------+---------+
|AIRLINE |MONTH|avg_departure_delay |n_flights|
+--------+-----+--------------------+---------+
| UA     |  12 | 18.7               | 12034   |
| DL     |   7 | 16.1               |  9540   |
```

In [ ]:
# Excluimos cancelados y retrasos de salida nulos.
# CANCELLED puede venir como texto o como número: la conversión permite ambos.
vuelos_validos = df_v.filter(F.col("CANCELLED").cast("int") == 0)

resultado_1 = (vuelos_validos
    .filter(F.col("DEPARTURE_DELAY").isNotNull())
    .groupBy("AIRLINE", "MONTH")
    .agg(
        F.round(F.avg("DEPARTURE_DELAY"), 2).alias("avg_departure_delay"),
        F.round(F.stddev("DEPARTURE_DELAY"), 2).alias("std_departure_delay"),
        F.count("*").alias("n_flights")
    )
    .orderBy(F.desc("avg_departure_delay"))
)
resultado_1.show(20, truncate=False)

#### **Ejercicio 2 - Tasa de puntualidad por aeropuerto de origen**

Para cada `ORIGIN_AIRPORT`, calcula:

* `is_ontime` => vuelos con `ARRIVAL_DELAY <= 0`
* `pct_on_time` => % de vuelos con `ARRIVAL_DELAY <= 0` (y `CANCELLED='0'`)
* `avg_arr_delay`

Muestra top 10 aeropuertos con mayor `pct_on_time`.

**Importante**: tienes que usar Window para seleccionar `ORIGIN_AIRPORT`.

**Resultado esperado (formato + ejemplo):**
```
+---------------+-----------+--------------+----------+
|ORIGIN_AIRPORT |pct_on_time|avg_arr_delay |  count   |
+---------------+-----------+--------------+----------+
| BOI           | 87.2      | -1.3         |   1523   |
| SLC           | 86.5      |  0.8         |   3891   |
...
```

In [ ]:
# La puntualidad solo se mide en vuelos no cancelados y con llegada registrada.
vuelos_llegada = (df_v
    .filter(F.col("CANCELLED").cast("int") == 0)
    .filter(F.col("ARRIVAL_DELAY").isNotNull())
    .withColumn("is_ontime", F.when(F.col("ARRIVAL_DELAY") <= 0, 1).otherwise(0))
)

# Ventana por aeropuerto; obtenemos métricas por grupo sin perder inicialmente las filas.
w_origen = Window.partitionBy("ORIGIN_AIRPORT")
w_orden = Window.partitionBy("ORIGIN_AIRPORT").orderBy(
    F.col("ARRIVAL_DELAY").asc(), F.col("FLIGHT_NUMBER").asc()
)

resultado_2 = (vuelos_llegada
    .withColumn("pct_on_time", F.round(100 * F.avg("is_ontime").over(w_origen), 2))
    .withColumn("avg_arr_delay", F.round(F.avg("ARRIVAL_DELAY").over(w_origen), 2))
    .withColumn("count", F.count("*").over(w_origen))
    .withColumn("fila", F.row_number().over(w_orden))
    .filter(F.col("fila") == 1)
    .select("ORIGIN_AIRPORT", "pct_on_time", "avg_arr_delay", "count")
    .orderBy(F.desc("pct_on_time"), F.desc("count"))
    .limit(10)
)
resultado_2.show(truncate=False)

#### **Ejercicio 3 - Rutas más concurridas y con mayor % de cancelación**

Agrupa por (`ORIGIN_AIRPORT`, `DESTINATION_AIRPORT`), calcula `num_flights`, `%cancelled` y ordena por `num_flights` descendente. Devuelve top 15.

Además, crea un segundo dataset con top 15, ordenado por `%cancelled` (con al menos 200 vuelos).

**Resultado esperado (formato + ejemplo):**

```
top_busy:
+--------------+-------------------+-----------+-------------+
|ORIGIN_AIRPORT|DESTINATION_AIRPORT|num_flights|pct_cancelled|
+--------------+-------------------+-----------+-------------+
| ATL          | MCO               |   14560   |    1.9      |
| LAX          | SFO               |   13840   |    2.3      |
...

top_cancel:
+--------------+-------------------+-----------+-------------+
|ORD           |EWR                |    2200   |    5.7      |
|JFK           |BOS                |    1850   |    5.2      |
...
```

In [ ]:
rutas = (df_v
    .groupBy("ORIGIN_AIRPORT", "DESTINATION_AIRPORT")
    .agg(
        F.count("*").alias("num_flights"),
        F.round(100 * F.avg(F.col("CANCELLED").cast("double")), 2).alias("pct_cancelled")
    )
)

top_busy = rutas.orderBy(F.desc("num_flights")).limit(15)
top_cancel = (rutas
    .filter(F.col("num_flights") >= 200)
    .orderBy(F.desc("pct_cancelled"), F.desc("num_flights"))
    .limit(15)
)
print("Top 15 rutas por número de vuelos:")
top_busy.show(truncate=False)
print("Top 15 rutas por porcentaje de cancelación (mínimo 200 vuelos):")
top_cancel.show(truncate=False)

#### **Ejercicio 4: Enriquecer con tabla de aerolíneas y top distancia media**

En base al siguiente *lookup* de nombres de aerolínea (simulado), haz `join` por `AIRLINE`. Calcula la **distancia media** por aerolínea (no cancelados) y ordénalo desc.

```python
airlines_lookup = spark.createDataFrame(
    [("AA","American Airlines"),("DL","Delta Air Lines"),("UA","United Airlines")],
    ["AIRLINE","AIRLINE_NAME"]
)
```

**Resultado esperado (formato + ejemplo):**

```
+-------+-------------------+------------+------+
|AIRLINE|AIRLINE_NAME       |avg_distance|n     |
+-------+-------------------+------------+------+
| UA    | United Airlines   | 1243.5     | 18234|
| AA    | American Airlines | 1198.2     | 21567|
| DL    | Delta Air Lines   | 1104.7     | 20780|
```

---

In [ ]:
airlines_lookup = spark.createDataFrame(
    [("AA", "American Airlines"), ("DL", "Delta Air Lines"), ("UA", "United Airlines")],
    ["AIRLINE", "AIRLINE_NAME"]
)

resultado_4 = (df_v
    .filter(F.col("CANCELLED").cast("int") == 0)
    .join(airlines_lookup, on="AIRLINE", how="inner")
    .groupBy("AIRLINE", "AIRLINE_NAME")
    .agg(
        F.round(F.avg("DISTANCE"), 2).alias("avg_distance"),
        F.count("*").alias("n")
    )
    .orderBy(F.desc("avg_distance"))
)
resultado_4.show(truncate=False)

#### **Ejercicio 5 - Top 5 destinos por aerolínea (lista ordenada)**

Para cada `AIRLINE`, construye una **lista ordenada** con los 5 destinos más frecuentes en `DESTINATION_AIRPORT`. Usa `collect_list` + `sort_array` con `struct`.

**Resultado esperado (formato + ejemplo):**

```
+-------+---------------------------+
|AIRLINE|top5_destinations          |
+-------+---------------------------+
|AA     |[DFW, ORD, MIA, LAX, CLT] |
|DL     |[ATL, DTW, MSP, LGA, JFK] |
...
```

In [ ]:
# Primero calculamos frecuencia por aerolínea y destino.
frecuencias = (df_v
    .filter(F.col("DESTINATION_AIRPORT").isNotNull())
    .groupBy("AIRLINE", "DESTINATION_AIRPORT")
    .agg(F.count("*").alias("frecuencia"))
)

# sort_array ordena ascendentemente el struct: -frecuencia sitúa primero los más frecuentes.
# El código del aeropuerto resuelve empates de forma determinista.
resultado_5 = (frecuencias
    .groupBy("AIRLINE")
    .agg(F.sort_array(F.collect_list(F.struct(
        (-F.col("frecuencia")).alias("orden"),
        F.col("DESTINATION_AIRPORT").alias("destino")
    ))).alias("destinos_ordenados"))
    .select("AIRLINE", F.expr("transform(slice(destinos_ordenados, 1, 5), x -> x.destino)")
            .alias("top5_destinations"))
    .orderBy("AIRLINE")
)
resultado_5.show(truncate=False)

#### **Ejercicio 6 - Clasificar vuelos por distancia y *pivot* por mes**

Define un UDF `distance_bucket(DISTANCE)` que devuelva `short` (<500), `medium` (500–1499) o `long` (≥1500). Calcula el **reparto** por `MONTH` y bucket (pivot), con recuentos.

**Resultado esperado (formato + ejemplo):**

```
+-----+-----+------+------+
|MONTH|short|medium| long |
+-----+-----+------+------+
|  1  | 4200| 8300 | 2100 |
|  2  | 3980| 7900 | 1985 |
...
```

In [ ]:
# UDF solicitada en el enunciado.
@F.udf(returnType=T.StringType())
def distance_bucket(distancia):
    if distancia is None:
        return None
    if distancia < 500:
        return "short"
    if distancia < 1500:
        return "medium"
    return "long"

resultado_6 = (df_v
    .withColumn("bucket", distance_bucket(F.col("DISTANCE")))
    .filter(F.col("bucket").isNotNull())
    .groupBy("MONTH")
    .pivot("bucket", ["short", "medium", "long"])
    .count()
    .fillna(0, subset=["short", "medium", "long"])
    .orderBy("MONTH")
)
resultado_6.show(12, truncate=False)

# En producción, when/otherwise suele rendir mejor que una UDF Python
# porque conserva la optimización del motor Spark.

#### **Ejercicio Extra - Causas de retraso: mayor causa por vuelo**

Crea array de pares `(delay_minutes, causa)` con `AIR_SYSTEM_DELAY, SECURITY_DELAY, AIRLINE_DELAY, LATE_AIRCRAFT_DELAY, WEATHER_DELAY`. Obtén la **causa con más minutos** por vuelo.

**Resultado esperado (formato + ejemplo):**

```
+-------------+-------------+---------------+--------------+
+FLIGHT_NUMBER+ARRIVAL_DELAY+TOP_DELAY_CAUSE+TOP_DELAY_MINS+
+-------------+-------------+---------------+--------------+
|  1234       |   85        | LATE_AIRCRAFT | 70           |
|   987       |   42        | WEATHER       | 40           |
...
```

In [ ]:
# Cada elemento del array es un struct (minutos, nombre de causa).
# Se convierten los nulos a cero antes de comparar los retrasos.
causas = [
    ("AIR_SYSTEM_DELAY", "AIR_SYSTEM"),
    ("SECURITY_DELAY", "SECURITY"),
    ("AIRLINE_DELAY", "AIRLINE"),
    ("LATE_AIRCRAFT_DELAY", "LATE_AIRCRAFT"),
    ("WEATHER_DELAY", "WEATHER")
]

pares = [F.struct(
    F.coalesce(F.col(columna).cast("double"), F.lit(0.0)).alias("delay_minutes"),
    F.lit(nombre).alias("causa")
) for columna, nombre in causas]

con_causas = df_v.withColumn("causas", F.array(*pares))
# Orden descendente por minutos; si empatan, se aplica un desempate lexicográfico.
resultado_7 = (con_causas
    .withColumn("mayor", F.element_at(F.sort_array(F.col("causas"), asc=False), 1))
    .select(
        "FLIGHT_NUMBER", "ARRIVAL_DELAY",
        F.when(F.col("mayor.delay_minutes") > 0, F.col("mayor.causa"))
         .otherwise(F.lit(None).cast("string")).alias("TOP_DELAY_CAUSE"),
        F.col("mayor.delay_minutes").alias("TOP_DELAY_MINS")
    )
)
resultado_7.show(20, truncate=False)
# Sin minutos positivos en ninguna causa, TOP_DELAY_CAUSE queda en NULL.